# Economic Data: IRS SOI and BLS QCEW

This notebook demonstrates the `siege_utilities.economic` package for accessing
federal economic datasets: IRS Statistics of Income (ZIP-code-level income data)
and BLS Quarterly Census of Employment and Wages.

## What this covers
1. Download IRS SOI data by state and tax year
2. Parse and normalize ZIP codes / FIPS codes
3. URL construction patterns for both IRS and BLS
4. Caching: avoid re-downloading on repeated runs

## 1. IRS Statistics of Income (ZIP-code level)

The `IRSSOIFiles` class downloads CSV files from IRS.gov containing income
statistics aggregated by ZIP code.

In [1]:
from siege_utilities.economic.irs.soi import IRSSOIFiles, DEFAULT_CACHE_DIR

# Show the default cache location
print(f"Default cache directory: {DEFAULT_CACHE_DIR}")

Default cache directory: /Users/dheerajchand/.siege_utilities/cache/irs_soi


In [2]:
# Construct the download URL for Texas, tax year 2020
files = IRSSOIFiles.__new__(IRSSOIFiles)
url = files.url_for(2020, "tx")
print(f"URL: {url}")

# The URL pattern is: https://www.irs.gov/pub/irs-soi/{year}zpallagi{state}.csv

URL: https://www.irs.gov/pub/irs-soi/2020zpallagitx.csv


### Parse normalization

IRS CSV files have ZIPCODE and STATEFIPS columns that may be missing leading zeros.
The parser zero-pads ZIPCODE to 5 digits and STATEFIPS to 2 digits, essential for
downstream spatial joins.

In [3]:
import tempfile
import csv
from pathlib import Path

# Create sample data to demonstrate parse normalization
tmp = Path(tempfile.mkdtemp())
sample_csv = tmp / "sample_soi.csv"
with open(sample_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['ZIPCODE', 'STATEFIPS', 'N1', 'A00100'])
    w.writeheader()
    w.writerows([
        {'ZIPCODE': '1234', 'STATEFIPS': '6', 'N1': '100', 'A00100': '5000'},
        {'ZIPCODE': '78701', 'STATEFIPS': '48', 'N1': '200', 'A00100': '12000'},
        {'ZIPCODE': '2101', 'STATEFIPS': '25', 'N1': '150', 'A00100': '8000'},
    ])

# Parse: zero-pads ZIPCODE to 5 digits, STATEFIPS to 2 digits
files = IRSSOIFiles(cache_dir=tmp)
df = files.parse(sample_csv)
print(df[['ZIPCODE', 'STATEFIPS', 'N1', 'A00100']].to_string(index=False))

ZIPCODE STATEFIPS  N1  A00100
  01234        06 100    5000
  78701        48 200   12000
  02101        25 150    8000


## 2. BLS Quarterly Census of Employment and Wages

The `QCEWFiles` class follows the same pattern for BLS QCEW data.

In [4]:
from siege_utilities.economic.bls.qcew import QCEWFiles

# QCEWFiles follows the same pattern: cache_dir, url construction, download, parse
files = QCEWFiles.__new__(QCEWFiles)
# Show the URL pattern for QCEW data
print(f"QCEWFiles class: {QCEWFiles}")
print(f"Both classes follow identical patterns: cache_dir, url_for(), download(), parse(), load()")

QCEWFiles class: <class 'siege_utilities.economic.bls.qcew.QCEWFiles'>
Both classes follow identical patterns: cache_dir, url_for(), download(), parse(), load()


## Key patterns

Both `IRSSOIFiles` and `QCEWFiles` follow the same architecture:

| Method | Purpose |
|--------|--------|
| `url_for(year, ...)` | Construct the download URL |
| `download(year, ...)` | Download to cache (skip if cached) |
| `parse(csv_path)` | Normalize column types/padding |
| `load(year, ...)` | Combines download + parse |

Cache prevents redundant downloads. Parse normalizes FIPS/ZIP codes for joins.